In [1]:
!pip -q install --upgrade numpy==1.26.4 scipy==1.11.4
!pip -q install mne==1.3.0 tqdm scikit-learn matplotlib



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os, json, time, random, warnings
from typing import Dict, List, Any

import numpy as np
import scipy.io as sio
import mne

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

PROJECT_ROOT = Path("/workspace/NeuroXAI")
DATA_RAW = PROJECT_ROOT / "data_raw"
EDF_DIR = DATA_RAW / "eeg"
ANNOT_PATH = DATA_RAW / "annotations" / "annotations_2017.mat"

DATA_CACHE = PROJECT_ROOT / "data_cache"
OUT_DIR = PROJECT_ROOT / "outputs"
SPLITS_DIR = OUT_DIR / "splits"
FIG_DIR = OUT_DIR / "figures"
LOG_DIR = OUT_DIR / "logs"
CKPT_DIR = OUT_DIR / "checkpoints"

for d in [DATA_CACHE, OUT_DIR, SPLITS_DIR, FIG_DIR, LOG_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert EDF_DIR.exists(), f"EDF folder missing: {EDF_DIR}"
assert ANNOT_PATH.exists(), f"Annotation file missing: {ANNOT_PATH}"

print("EDF_DIR:", EDF_DIR)
print("ANNOT_PATH:", ANNOT_PATH)
print("DATA_CACHE:", DATA_CACHE)


EDF_DIR: /workspace/NeuroXAI/data_raw/eeg
ANNOT_PATH: /workspace/NeuroXAI/data_raw/annotations/annotations_2017.mat
DATA_CACHE: /workspace/NeuroXAI/data_cache


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

# Preprocessing configuration
SFREQ_TARGET = 100
L_FREQ, H_FREQ = 0.5, 30.0

WIN_SEC = 10
STEP_SEC_SEIZURE = 5
STEP_SEC_NONSEIZURE = 10

SEIZURE_WINDOW_RATIO = 0.25
MIN_RECORDING_SECONDS = 900
MIN_SEIZURE_SECONDS = 1
CONSENSUS_THRESHOLD = 2


In [4]:
BIPOLAR_PAIRS = [
    ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),
    ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),
    ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),
    ("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),
    ("Fz", "Cz"), ("Cz", "Pz"),
]
BIPOLAR_NAMES = [f"{a}-{b}" for a, b in BIPOLAR_PAIRS]
CANON_ELECTRODES = sorted(list(set([x for p in BIPOLAR_PAIRS for x in p])))

def _canonicalize_ch_name(ch: str) -> str:
    c = ch.strip()
    if c.lower().startswith("eeg "):
        c = c[4:].strip()
    for suf in ["-REF", "-Ref", "-ref"]:
        if c.endswith(suf):
            c = c[:-len(suf)].strip()
    return c

def normalize_and_pick_eeg(raw: mne.io.BaseRaw):
    raw = raw.copy()
    raw.rename_channels({ch: _canonicalize_ch_name(ch) for ch in raw.ch_names})
    raw.set_montage("standard_1020", on_missing="ignore")
    raw.pick_types(eeg=True)

    missing = sorted(list(set(CANON_ELECTRODES) - set(raw.ch_names)))
    if missing:
        print(f"[WARN] Missing electrodes: {missing}")
        return None

    raw.pick_channels(CANON_ELECTRODES)
    return raw


In [5]:
def load_annotat_new(mat_path: str) -> np.ndarray:
    mat = sio.loadmat(mat_path)
    if "annotat_new" not in mat:
        raise KeyError(f"'annotat_new' not found. Keys={list(mat.keys())}")
    return mat["annotat_new"]

def get_baby_seconds_matrix(annotat_new: np.ndarray, baby_id: int) -> np.ndarray:
    i = baby_id - 1
    if annotat_new.shape[0] == 79:
        cell = annotat_new[i, 0]
    elif annotat_new.shape[1] == 79:
        cell = annotat_new[0, i]
    else:
        raise RuntimeError(f"Unexpected annotat_new shape: {annotat_new.shape}")

    A = np.array(cell)
    if A.ndim == 1:
        A = np.vstack([np.array(x).squeeze() for x in A])

    if A.shape[0] != 3 and A.shape[1] == 3:
        A = A.T
    if A.shape[0] != 3:
        raise RuntimeError(f"Expected 3 experts, got {A.shape}")

    A = np.nan_to_num(A)
    return np.asarray(A, dtype=np.float32)

def consensus_1hz_labels(A_3xT: np.ndarray) -> np.ndarray:
    A = (A_3xT > 0.5).astype(np.int64)
    labels = (A.sum(axis=0) >= CONSENSUS_THRESHOLD).astype(np.int64)
    return labels

annotat_new = load_annotat_new(str(ANNOT_PATH))
print("Loaded annotat_new shape:", annotat_new.shape)

def infer_baby_id_from_filename(p: Path) -> int:
    digits = "".join([c if c.isdigit() else " " for c in p.stem]).split()
    if not digits:
        raise RuntimeError(f"Cannot infer baby_id from filename: {p.name}")
    bid = int(digits[-1])
    if not (1 <= bid <= 79):
        raise RuntimeError(f"baby_id out of range from {p.name}: {bid}")
    return bid


Loaded annotat_new shape: (1, 79)


In [6]:
def preprocess_one(edf_path: Path, baby_id: int):
    t0 = time.time()

    raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose="ERROR")
    raw = normalize_and_pick_eeg(raw)
    if raw is None:
        raise RuntimeError(f"Patient {baby_id}: required electrodes missing")

    raw.filter(L_FREQ, H_FREQ, fir_design="firwin", verbose="ERROR")
    raw.resample(SFREQ_TARGET, npad="auto")

    raw = mne.set_bipolar_reference(
        raw,
        anode=[a for a, b in BIPOLAR_PAIRS],
        cathode=[b for a, b in BIPOLAR_PAIRS],
        ch_name=BIPOLAR_NAMES,
        drop_refs=True,
        copy=False,
    )

    if raw.get_data().shape[0] != 18:
        raise RuntimeError(f"Patient {baby_id}: bipolar montage did not produce 18 channels")
    if raw.ch_names != BIPOLAR_NAMES:
        raise RuntimeError(f"Patient {baby_id}: bipolar channel order mismatch")

    data = raw.get_data().astype(np.float32)
    total_seconds = data.shape[1] // SFREQ_TARGET
    if total_seconds < MIN_RECORDING_SECONDS:
        raise RuntimeError(f"Patient {baby_id}: total_seconds={total_seconds} < {MIN_RECORDING_SECONDS}")

    data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)

    A = get_baby_seconds_matrix(annotat_new, baby_id)
    y_sec = consensus_1hz_labels(A)

    T = min(len(y_sec), total_seconds)
    if T <= 0:
        raise RuntimeError(f"Patient {baby_id}: alignment T<=0")

    y_sec = y_sec[:T]
    data = data[:, :T * SFREQ_TARGET]
    total_seconds = T
    seizure_seconds = int(y_sec.sum())

    win_samp = WIN_SEC * SFREQ_TARGET
    step_samp_seiz = STEP_SEC_SEIZURE * SFREQ_TARGET
    step_samp_non = STEP_SEC_NONSEIZURE * SFREQ_TARGET

    X_list, y_list = [], []
    st = 0
    max_start = data.shape[1] - win_samp

    while st <= max_start:
        ed = st + win_samp
        xw = data[:, st:ed]

        sec_st = st // SFREQ_TARGET
        sec_ed = min(ed // SFREQ_TARGET, len(y_sec))
        seg = y_sec[sec_st:sec_ed]

        if len(seg) == 0:
            st += step_samp_non
            continue

        yw = 1 if (seg.mean() >= SEIZURE_WINDOW_RATIO) else 0
        X_list.append(xw)
        y_list.append(yw)

        if yw == 1:
            st += step_samp_seiz
        else:
            st += step_samp_non

    if not X_list:
        raise RuntimeError(f"Patient {baby_id}: 0 windows")

    X = np.stack(X_list, axis=0).astype(np.float32)
    y = np.asarray(y_list, dtype=np.int64)

    seizure_windows = int(y.sum())
    total_windows = int(len(y))
    seizure_pct = 100.0 * seizure_windows / max(1, total_windows)

    if seizure_seconds > 0 and seizure_windows == 0:
        raise RuntimeError(f"Patient {baby_id}: seizure_seconds>0 but seizure_windows==0")

    print(
        f"[PATIENT {baby_id:02d}] total_seconds={total_seconds} | "
        f"seizure_seconds={seizure_seconds} | "
        f"total_windows={total_windows} | "
        f"seizure_windows={seizure_windows} | "
        f"seizure_percentage={seizure_pct:.2f}% | "
        f"time={time.time() - t0:.2f}s"
    )

    meta = {
        "patient_id": baby_id,
        "total_seconds": total_seconds,
        "seizure_seconds": seizure_seconds,
        "total_windows": total_windows,
        "seizure_windows": seizure_windows,
        "seizure_window_ratio": SEIZURE_WINDOW_RATIO,
        "win_sec": WIN_SEC,
        "step_sec_seizure": STEP_SEC_SEIZURE,
        "step_sec_nonseizure": STEP_SEC_NONSEIZURE,
        "sfreq_target": SFREQ_TARGET,
        "l_freq": L_FREQ,
        "h_freq": H_FREQ,
    }

    return X, y, meta


In [7]:
# Delete old cache first if you changed preprocessing settings.
for p in DATA_CACHE.glob("patient_*"):
    if p.is_file():
        p.unlink()

for name in ["patient_meta.json", "preprocessing_config.json"]:
    fp = DATA_CACHE / name
    if fp.exists():
        fp.unlink()

patient_meta = {}
failed = []

print("[INFO] Caching to .npy")
for p in sorted(EDF_DIR.glob("*.edf")):
    try:
        pid = infer_baby_id_from_filename(p)
        X, y, meta = preprocess_one(p, pid)

        np.save(DATA_CACHE / f"patient_{pid:02d}_X.npy", X)
        np.save(DATA_CACHE / f"patient_{pid:02d}_y.npy", y)
        with open(DATA_CACHE / f"patient_{pid:02d}_meta.json", "w") as f:
            json.dump(meta, f, indent=2)
        patient_meta[pid] = meta
    except Exception as e:
        failed.append((str(p), str(e)))

with open(DATA_CACHE / "patient_meta.json", "w") as f:
    json.dump({str(k): v for k, v in patient_meta.items()}, f, indent=2)

with open(DATA_CACHE / "preprocessing_config.json", "w") as f:
    json.dump(
        {
            "sfreq_target": SFREQ_TARGET,
            "l_freq": L_FREQ,
            "h_freq": H_FREQ,
            "win_sec": WIN_SEC,
            "step_sec_seizure": STEP_SEC_SEIZURE,
            "step_sec_nonseizure": STEP_SEC_NONSEIZURE,
            "seizure_window_ratio": SEIZURE_WINDOW_RATIO,
            "min_recording_seconds": MIN_RECORDING_SECONDS,
            "min_seizure_seconds": MIN_SEIZURE_SECONDS,
            "bipolar_names": BIPOLAR_NAMES,
        },
        f,
        indent=2,
    )

print("\nDONE preprocessing + caching.")
print("Cached patients:", len(patient_meta))
print("Failures:", len(failed))
if failed:
    for fp, err in failed[:20]:
        print(fp, "\n ", err, "\n")


[INFO] Caching to .npy
[PATIENT 01] total_seconds=6993 | seizure_seconds=1543 | total_windows=860 | seizure_windows=323 | seizure_percentage=37.56% | time=3.22s
[PATIENT 10] total_seconds=5427 | seizure_seconds=0 | total_windows=542 | seizure_windows=0 | seizure_percentage=0.00% | time=2.50s
[PATIENT 11] total_seconds=7488 | seizure_seconds=94 | total_windows=758 | seizure_windows=20 | seizure_percentage=2.64% | time=2.50s
[PATIENT 12] total_seconds=4468 | seizure_seconds=0 | total_windows=446 | seizure_windows=0 | seizure_percentage=0.00% | time=1.90s
[PATIENT 13] total_seconds=15416 | seizure_seconds=1367 | total_windows=1679 | seizure_windows=275 | seizure_percentage=16.38% | time=6.38s
[PATIENT 14] total_seconds=3726 | seizure_seconds=2280 | total_windows=608 | seizure_windows=471 | seizure_percentage=77.47% | time=1.35s
[PATIENT 15] total_seconds=6898 | seizure_seconds=1282 | total_windows=823 | seizure_windows=268 | seizure_percentage=32.56% | time=2.86s
[PATIENT 16] total_second

In [8]:
with open(DATA_CACHE / "patient_meta.json", "r") as f:
    patient_meta = json.load(f)

patient_ids = sorted([int(k) for k in patient_meta.keys()])
seizure_rich_ids = sorted([pid for pid in patient_ids if patient_meta[str(pid)]["seizure_seconds"] >= MIN_SEIZURE_SECONDS])
seizure_free_ids = sorted([pid for pid in patient_ids if patient_meta[str(pid)]["seizure_seconds"] == 0])

print("Seizure-rich patients:", len(seizure_rich_ids), seizure_rich_ids)
print("Seizure-free patients:", len(seizure_free_ids), seizure_free_ids)


Seizure-rich patients: 46 [1, 4, 5, 7, 9, 11, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 25, 31, 33, 34, 36, 38, 39, 40, 41, 44, 47, 50, 51, 52, 54, 62, 63, 64, 66, 67, 68, 69, 71, 73, 74, 75, 76, 77, 78, 79]
Seizure-free patients: 33 [2, 3, 6, 8, 10, 12, 18, 24, 26, 27, 28, 29, 30, 32, 35, 37, 42, 43, 45, 46, 48, 49, 53, 55, 56, 57, 58, 59, 60, 61, 65, 70, 72]


In [9]:
def split_patients(ids, seed=42, train_ratio=0.70, val_ratio=0.10):
    rng = np.random.default_rng(seed)
    ids = ids.copy()
    rng.shuffle(ids)

    n = len(ids)
    n_train = max(1, int(round(train_ratio * n)))
    n_val = max(1, int(round(val_ratio * n)))
    if n_train + n_val >= n:
        n_val = max(1, n - n_train - 1)

    train = ids[:n_train]
    val = ids[n_train:n_train + n_val]
    test = ids[n_train + n_val:]
    return sorted(train), sorted(val), sorted(test)

# Realistic split: rich + seizure-free mixed
rich_train, rich_val, rich_test = split_patients(seizure_rich_ids, seed=42)
free_train, free_val, free_test = split_patients(seizure_free_ids, seed=42)

train_ids = sorted(rich_train + free_train)
val_ids = sorted(rich_val + free_val)
test_ids = sorted(rich_test + free_test)

dev_splits = {
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids,
    "rich_train_ids": rich_train,
    "rich_val_ids": rich_val,
    "rich_test_ids": rich_test,
    "free_train_ids": free_train,
    "free_val_ids": free_val,
    "free_test_ids": free_test,
}

with open(SPLITS_DIR / "patient_splits_dev.json", "w") as f:
    json.dump(dev_splits, f, indent=2)

print("=== DEVELOPMENT SPLIT (ALL PATIENTS) ===")
print("TRAIN:", train_ids)
print("VAL  :", val_ids)
print("TEST :", test_ids)
print("Rich train/val/test:", len(rich_train), len(rich_val), len(rich_test))
print("Free train/val/test:", len(free_train), len(free_val), len(free_test))


=== DEVELOPMENT SPLIT (ALL PATIENTS) ===
TRAIN: [2, 7, 8, 9, 11, 12, 13, 14, 16, 17, 18, 23, 24, 25, 27, 28, 30, 31, 33, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 66, 69, 71, 72, 73, 74, 75, 76, 77, 79]
VAL  : [1, 19, 29, 34, 35, 39, 49, 67]
TEST : [3, 4, 5, 6, 10, 15, 20, 21, 22, 26, 32, 64, 65, 68, 70, 78]
Rich train/val/test: 32 5 9
Free train/val/test: 23 3 7


In [10]:
# Benchmark split: seizure-rich only
bench_train_ids, bench_val_ids, bench_test_ids = split_patients(seizure_rich_ids, seed=42)

benchmark_splits = {
    "train_ids": bench_train_ids,
    "val_ids": bench_val_ids,
    "test_ids": bench_test_ids,
    "mode": "seizure_rich_only"
}

with open(SPLITS_DIR / "patient_splits_benchmark.json", "w") as f:
    json.dump(benchmark_splits, f, indent=2)

print("=== BENCHMARK SPLIT (SEIZURE-RICH ONLY) ===")
print("TRAIN:", bench_train_ids)
print("VAL  :", bench_val_ids)
print("TEST :", bench_test_ids)


=== BENCHMARK SPLIT (SEIZURE-RICH ONLY) ===
TRAIN: [7, 9, 11, 13, 14, 16, 17, 23, 25, 31, 33, 36, 38, 40, 41, 44, 47, 50, 51, 52, 54, 62, 63, 66, 69, 71, 73, 74, 75, 76, 77, 79]
VAL  : [1, 19, 34, 39, 67]
TEST : [4, 5, 15, 20, 21, 22, 64, 68, 78]


In [11]:
def build_loso_folds(seizure_rich_ids, seizure_free_ids, seed=42):
    rng = np.random.default_rng(seed)
    folds = []

    for test_id in sorted(seizure_rich_ids):
        rich_remaining = [pid for pid in seizure_rich_ids if pid != test_id]
        rich_remaining = rich_remaining.copy()
        rng.shuffle(rich_remaining)

        n_val_rich = max(1, int(round(0.10 * len(rich_remaining))))
        val_rich = sorted(rich_remaining[:n_val_rich])
        train_rich = sorted(rich_remaining[n_val_rich:])

        free_ids = seizure_free_ids.copy()
        rng.shuffle(free_ids)
        n_val_free = max(1, int(round(0.10 * len(free_ids))))
        val_free = sorted(free_ids[:n_val_free])
        train_free = sorted(free_ids[n_val_free:])

        folds.append({
            "test_id": test_id,
            "train_ids": sorted(train_rich + train_free),
            "val_ids": sorted(val_rich + val_free),
            "train_rich_ids": train_rich,
            "val_rich_ids": val_rich,
            "train_free_ids": train_free,
            "val_free_ids": val_free,
        })
    return folds

loso_folds = build_loso_folds(seizure_rich_ids, seizure_free_ids, seed=42)
with open(SPLITS_DIR / "patient_splits_loso.json", "w") as f:
    json.dump(
        {
            "seizure_rich_ids": seizure_rich_ids,
            "seizure_free_ids": seizure_free_ids,
            "folds": loso_folds,
        },
        f,
        indent=2,
    )

print("=== LOSO FOLDS ===")
print("Number of folds:", len(loso_folds))
print("Example fold 1:", loso_folds[0] if len(loso_folds) > 0 else "None")


=== LOSO FOLDS ===
Number of folds: 46
Example fold 1: {'test_id': 1, 'train_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78], 'val_ids': [35, 41, 48, 54, 67, 70, 79], 'train_rich_ids': [4, 5, 7, 9, 11, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 25, 31, 33, 34, 36, 38, 39, 40, 44, 47, 50, 51, 52, 62, 63, 64, 66, 68, 69, 71, 73, 74, 75, 76, 77, 78], 'val_rich_ids': [41, 54, 67, 79], 'train_free_ids': [2, 3, 6, 8, 10, 12, 18, 24, 26, 27, 28, 29, 30, 32, 37, 42, 43, 45, 46, 49, 53, 55, 56, 57, 58, 59, 60, 61, 65, 72], 'val_free_ids': [35, 48, 70]}
